# Slide Analyzer

*You got a slide deck from the course and are in front of a slide, staring at a sentence you don't fully grasp... or of which you want to know more!*

This app is for you! It analyzes a slide, identifies relevant topics and gives you more information. You can then save it to your computer.

## How to use it

Upload a **slide deck (PDF)** — or the image of a single slide — browse its slides, select one and press
**Analyze slide**; the app:

1. **Structure** – finds the slide title *and* splits the rest of the slide into relevant areas (bounding boxes), in a single model call
2. **Topics & relations** – identifies the main topic of each area and how the areas relate to each other
3. **Visualization** – draws the slide with the boxes (and, optionally, arrows for the relations)
4. **Details on click** – clicking a box asks the model for an in-depth explanation of that area; when analyzing a box, you can ask for specific information, even adding a custom prompt.
5. **Download report** - saves everything analyzed so far as one self-contained HTML file (open it in a browser and use *Print → Save as PDF* to get a PDF).

## Technical details
Analyses are cached per slide: going back to a slide that was already analyzed costs no API call.
Every step is one call to an OpenAI vision model using **Structured Outputs**: the response is validated
against a Pydantic schema, so each step returns a well-defined JSON object that the next step (and the drawing code) consumes.

Each step uses its own model and reasoning effort (see `STEP_CONFIG`): `gpt-6.1-sol` for the two
analysis steps, and the faster `gpt-6-luna` with **web search** for the details shown on click.

```
PDF ─► thumbnails ─► select slide ─► render page
                                         │
            [Analyze slide] ─► [1] SlideStructure (title + areas) ─► [2] TopicsResult ─► draw boxes
                                                                             │
                                                    click on box ─► [3] AreaDetail (cached per area)
```

## 0. Setup

In [ ]:
# %pip install -q "openai>=2" "gradio>=5" "pillow>=10.1" "pydantic>=2" "pypdfium2>=4" markdown-it-py linkify-it-py

import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

requirements = [
    ("openai", "2.0.0"),
    ("gradio", "5.0.0"),
    ("pillow", "10.1.0"),
    ("pydantic", "2.0.0"),
    ("pypdfium2", "4.0.0"),
    ("markdown-it-py", None),
    ("linkify-it-py", None)
]
missing_or_outdated = []
for pkg, min_version in requirements:
    try:
        current_version = version(pkg)
        
        if min_version and Version(current_version) < Version(min_version):
                missing_or_outdated.append(f"{pkg}>={min_version}")
            
    except PackageNotFoundError:
        missing_or_outdated.append(f"{pkg}>={min_version}" if min_version else pkg)
if missing_or_outdated:
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + missing_or_outdated)

In [ ]:
import base64          # encode images as base64 to embed them in the API request / in HTML
import html            # escape text inserted into the HTML of the details panel
import importlib.util  # check whether an optional package is installed
import io              # in-memory byte buffers (no temp files needed)
import json            # serialize previous results into the prompts
import math            # angles for drawing arrowheads
import os              # read API key / model name from environment variables
import tempfile        # folder for the downloadable report
from datetime import datetime   # timestamp in the report
import textwrap        # shorten long labels so they fit on the image
import threading       # lock around PDFium, which is not thread-safe
from getpass import getpass   # ask for the API key without echoing it
from typing import Literal, Optional

import gradio as gr
import pypdfium2 as pdfium    # PDF page -> image rendering
from markdown_it import MarkdownIt   # Markdown -> HTML for the details cards (installed with gradio)
from openai import OpenAI
from PIL import Image, ImageDraw, ImageFont, ImageOps
from pydantic import BaseModel, Field

in_gcolab = 'google.colab' in sys.modules

# The OpenAI client reads the key from the OPENAI_API_KEY environment variable.
# If it is not set, ask for it interactively (it is kept only in this kernel's memory).
if in_gcolab:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

# Model configuration PER STEP. OpenAI has no dedicated detection/segmentation model: image
# understanding and localization are capabilities of the general GPT models, so the "specialization"
# is done by choosing, for each step, the model tier, how much it reasons, and which tools it may use.
#
#   model      -> "gpt-6-astra" (most capable, most expensive), "gpt-6.1-sol" (near-Astra quality at
#                 lower cost), "gpt-6-luna" (cheapest and fastest). All accept images + structured outputs.
#   effort     -> reasoning effort: "none" (Luna only), "minimal", "low", "medium", "high", "xhigh", "max".
#                 More effort = more hidden "thinking" tokens: usually more accurate, but slower and pricier.
#   web_search -> lets the model search the web (built-in OpenAI tool) before answering.
STEP_CONFIG = {
    # Step 1 - title + bounding boxes: localization is the hardest part for language models,
    # so it gets the stronger model and more reasoning. Try "high" or "gpt-6-astra" if boxes are off.
    "structure": {"model": "gpt-6.1-sol", "effort": "medium", "web_search": False},
    # Step 2 - topics & relations: reading comprehension on a slide whose boxes are already drawn,
    # an easier task -> same model, little reasoning.
    "topics":    {"model": "gpt-6.1-sol", "effort": "low", "web_search": False},
    # Step 3 - details on click: the user is waiting, so speed matters -> the fast, cheap model,
    # plus web search so the background information is current and comes with sources.
    "detail":    {"model": "gpt-6-luna", "effort": "low", "web_search": True},
}

# A single client object is reused for all requests (it keeps the HTTP connection pool).
client = OpenAI()

## 1. JSON schemas of each step

Coordinates are **normalized to 0–1000** (`(0,0)` = top-left, `(1000,1000)` = bottom-right), so they
do not depend on the resolution of the uploaded image or of the copy sent to the API.

In [ ]:
# Each Pydantic class below is converted by the OpenAI SDK into a JSON Schema and sent with the
# request ("Structured Outputs"). The model is then forced to answer with JSON that matches the
# schema exactly, and the SDK parses it back into an instance of the class.
#
# Notes on Structured Outputs:
# - every field is required; "optional" values are expressed as `Optional[...]` (the model returns null)
# - `Literal[...]` becomes an enum, i.e. the model can only choose among the listed values
# - the `description` of each Field is sent to the model and acts as an extra, per-field instruction


class Box(BaseModel):
    """Bounding box in coordinates normalized to 0-1000.

    Normalized coordinates are used instead of pixels because the image sent to the API may be
    downscaled (see MAX_API_SIDE): with 0-1000 values the boxes stay valid for any resolution and
    are converted to pixels only when drawing (see `box_to_px`)."""
    x_min: int = Field(description="Left edge, 0-1000 (fraction of image width x 1000)")
    y_min: int = Field(description="Top edge, 0-1000 (fraction of image height x 1000)")
    x_max: int = Field(description="Right edge, 0-1000")
    y_max: int = Field(description="Bottom edge, 0-1000")


# --- Step 1: slide structure = title + content areas (ONE model call) ----------------
# Title and areas are requested together: the image is sent only once, which saves network
# traffic and input tokens (images are the most expensive part of the prompt), and the model
# sees the whole problem at once, so it naturally keeps the title out of the content areas.

class Area(BaseModel):
    """One semantically coherent block of content (bullet list, diagram, chart, ...)."""
    id: str = Field(description="Area id: A1, A2, ... in reading order")
    label: str = Field(description="Short description of what the area contains (max ~8 words)")
    # The enum gives the UI a consistent vocabulary and helps the model reason about the block.
    content_type: Literal["text", "bullet_list", "diagram", "chart", "table", "image", "equation", "code", "mixed"]
    box: Box


class SlideStructure(BaseModel):
    """Result of step 1: everything about the *geometry* of the slide."""
    title: str = Field(description="Slide title, verbatim as written on the slide")
    subtitle: Optional[str] = Field(description="Subtitle if present, otherwise null")
    # Some slides have no explicit title: in that case the model invents one and flags it.
    title_is_inferred: bool = Field(description="True if the slide has no explicit title and it was inferred")
    title_box: Optional[Box] = Field(description="Box around the title text, null if inferred")
    slide_language: str = Field(description="Main language of the slide text, e.g. 'English'")
    areas: list[Area] = Field(description="Content areas of the slide, title excluded, in reading order")


# --- Step 2: topics and relations ---------------------------------------------------
# This step is about *meaning*: what each area talks about and how areas are connected.

class Topic(BaseModel):
    area_id: str          # links the topic back to an Area of step 1
    main_topic: str = Field(description="Main topic of the area, 2-6 words")
    summary: str = Field(description="1-2 sentence summary of the area content")
    keywords: list[str]
    # Role of the area in the slide's argument (used in the overview table).
    role: Literal["introduction", "definition", "main_point", "supporting_detail", "example",
                  "data_evidence", "process_step", "comparison", "conclusion", "other"]


class Relation(BaseModel):
    """A directed edge of the 'topic graph': source area --relation_type--> target area."""
    source_area_id: str
    target_area_id: str
    relation_type: Literal["leads_to", "causes", "supports", "elaborates", "example_of", "part_of",
                           "contrasts_with", "compares_with", "depends_on", "precedes", "other"]
    description: str = Field(description="One sentence explaining the relation")


class TopicsResult(BaseModel):
    topics: list[Topic]        # one entry per area
    relations: list[Relation]  # zero or more edges between areas
    overall_message: str = Field(description="The key message of the whole slide in 1-2 sentences")


# --- Step 3: details of one area (requested only when the user clicks on it) ---------

class Connection(BaseModel):
    area_id: str          # the *other* area
    how_related: str      # free-text explanation of the link


class Source(BaseModel):
    title: str
    url: str


class AreaDetail(BaseModel):
    area_id: str
    topic: str
    extracted_text: str = Field(description="Text visible in the area, transcribed verbatim (empty if none)")
    explanation: str = Field(description="Clear explanation of the area content, 1-2 paragraphs")
    key_points: list[str]
    # Explicitly separated from the explanation, so the user can tell what is on the slide
    # and what is additional knowledge coming from the model.
    background: str = Field(description="Useful context NOT shown on the slide (definitions, history, pitfalls)")
    connections: list[Connection] = Field(description="How this area relates to the other areas")
    questions_to_explore: list[str] = Field(description="2-4 follow-up questions for a learner")
    # Filled from the web search results (see step3_detail); empty if web search is disabled.
    sources: list[Source] = Field(description="Web pages used for the background information")

## 2. OpenAI helper

One generic function sends text + images and parses the answer straight into a Pydantic model (`responses.parse` with `text_format`).

In [ ]:
# System prompt shared by all steps: it fixes the coordinate convention once, so the
# step-specific prompts can focus on the task.
SYSTEM_PROMPT = """You are an expert in the visual analysis of presentation slides.
All coordinates you return are integers normalized to 0-1000: (0,0) is the top-left corner
and (1000,1000) the bottom-right corner of the image. Bounding boxes must tightly enclose
the content they refer to. Answer in the language of the slide."""

# Images larger than this (longest side, in pixels) are downscaled before being sent.
# With detail="high" OpenAI internally resizes images anyway, so sending bigger images only
# wastes upload bandwidth without improving the result.
MAX_API_SIDE = 2048


def to_data_url(img: Image.Image) -> str:
    """Encode a PIL image as a base64 'data:' URL, the format accepted by the API for inline images."""
    img = img.copy()                               # never modify the caller's image
    img.thumbnail((MAX_API_SIDE, MAX_API_SIDE))    # downscale in place, keeping the aspect ratio
    buf = io.BytesIO()
    img.save(buf, format="PNG")                    # PNG is lossless: small text stays readable
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def call_structured(prompt: str, images: list[Image.Image], schema: type[BaseModel], step: str):
    """Send a prompt plus zero or more images and return the API response, with the answer
    parsed into `schema` available as `response.output_parsed`.

    Every step of the pipeline goes through this function, so the way the API is called
    (system prompt, image encoding) is defined in one place only; `step` selects the
    model / reasoning effort / tools of that step in STEP_CONFIG.
    The full response (not only the parsed object) is returned because step 3 also needs
    the web citations attached to the answer (see `url_citations`)."""
    cfg = STEP_CONFIG[step]
    # A user message is a list of "content parts": first the text, then the images.
    content = [{"type": "input_text", "text": prompt}]
    # detail="high" lets the model look at the image in tiles -> better at reading small text
    # and at locating boxes, at the cost of more input tokens.
    content += [{"type": "input_image", "image_url": to_data_url(im), "detail": "high"} for im in images]

    # Responses API + `text_format`: the SDK turns the Pydantic class into a strict JSON Schema,
    # the model is constrained to it, and the SDK validates and parses the result.
    # Structured outputs also work together with tools: the model may search the web first,
    # and its FINAL message is still forced to match the schema.
    response = client.responses.parse(
        model=cfg["model"],
        instructions=SYSTEM_PROMPT,   # = system message
        input=[{"role": "user", "content": content}],
        text_format=schema,
        reasoning={"effort": cfg["effort"]},
        tools=[{"type": "web_search"}] if cfg["web_search"] else [],
    )
    # output_parsed is None if the model refused or the output was cut off (e.g. token limit).
    if response.output_parsed is None:
        raise RuntimeError("The model did not return a valid structured answer (refusal or truncation).")
    return response


def url_citations(response) -> list[dict]:
    """Web pages the model actually cited, read from the `url_citation` annotations of the answer.

    These come from the web search tool itself, so unlike URLs written by the model in its JSON,
    they cannot be hallucinated. Returned as [{"title": ..., "url": ...}], without duplicates."""
    seen, out = set(), []
    for item in response.output:                      # reasoning items, web_search_call items, message
        if item.type != "message":
            continue
        for part in item.content:
            for ann in getattr(part, "annotations", None) or []:
                if ann.type == "url_citation" and ann.url not in seen:
                    seen.add(ann.url)
                    out.append({"title": ann.title or ann.url, "url": ann.url})
    return out

## 3. Pipeline steps

In [ ]:
def step1_structure(slide: Image.Image) -> SlideStructure:
    """Step 1: title AND content areas in a single request (the slide image is uploaded once)."""
    prompt = """Step 1 - SLIDE STRUCTURE.
A) TITLE: identify the title of this slide (and the subtitle, if any). Return the title verbatim
   and the bounding box around the title text. If the slide has no explicit title, infer a short
   one, set title_is_inferred=true and title_box=null.
B) AREAS: split the REST of the slide into 2-8 semantically coherent areas (e.g. a block of bullet
   points, a diagram, a chart, a picture with its caption, a column of a comparison, a callout box).
   Rules:
   - Exclude the title/subtitle, logos, footers, page numbers and purely decorative elements.
   - Areas must not overlap significantly and must together cover all meaningful content.
   - Number them A1, A2, ... in natural reading order."""
    return call_structured(prompt, [slide], SlideStructure, step="structure").output_parsed


def step2_topics(slide: Image.Image, structure: SlideStructure) -> TopicsResult:
    """Step 2: main topic of each area + relations between areas.

    Two images are sent: the clean slide (to read the content) and the same slide with the
    area boxes and ids drawn on it, so the model can unambiguously match ids to regions
    instead of having to interpret raw coordinates."""
    annotated = draw_annotations(slide, areas=[a.model_dump() for a in structure.areas])
    # The output of step 1 is passed back as JSON: the model reuses the ids and labels
    # instead of re-discovering the layout.
    prompt = f"""Step 2 - TOPICS AND RELATIONS.
Slide structure (areas are also drawn and labelled on the second image):
{structure.model_dump_json(indent=2)}

For EVERY area give its main topic, a short summary, keywords and its role in the slide.
Then list the meaningful relations between areas (directed: source -> target). Only include
relations that are actually supported by the slide (arrows, ordering, logic, shared concepts).
Finally state the overall message of the slide."""
    return call_structured(prompt, [slide, annotated], TopicsResult, step="topics").output_parsed


# --- What the user wants to know about a clicked area (menu "What would you like to know?") ---------
# A request is identified by a string: the menu label for the preset ones, "Custom: <text>" for the
# user's own question. It is part of the cache key of the answers, so asking something different
# about the same area produces a new answer (and a new card).
REQUEST_EXPLAIN = "Explain the content of the area"
REQUEST_RESOURCES = "Provide additional resources on the topic"
REQUEST_CUSTOM = "Custom"
REQUEST_CHOICES = [REQUEST_EXPLAIN, REQUEST_RESOURCES, REQUEST_CUSTOM]

REQUEST_PROMPTS = {   # instructions sent to the model for the preset requests ({a} = area id)
    REQUEST_EXPLAIN:
        "Explain area {a} in depth for a learner: transcribe its text, explain it, list the key points, "
        "add useful background knowledge that is NOT on the slide, and explain how it connects to the other areas.",
    REQUEST_RESOURCES:
        "Find additional resources to study the topic of area {a} in more depth (tutorials, textbook chapters, "
        "lecture notes, articles, videos, papers); prefer reliable and freely accessible ones. In `explanation`, "
        "briefly present the resources and what each one is useful for, and list all of them in `sources`. "
        "Also transcribe the text of the area; keep the other fields short.",
}


def request_key(choice: Optional[str], custom_text: Optional[str]) -> Optional[str]:
    """Request string for the menu choice (None if "Custom" was chosen but nothing was written)."""
    if choice == REQUEST_CUSTOM:
        text = (custom_text or "").strip()
        return f"Custom: {text}" if text else None
    return choice if choice in REQUEST_PROMPTS else REQUEST_EXPLAIN


def request_instructions(request: str, area_id: str) -> str:
    """Instructions for the model: the preset prompt, or the user's own question."""
    if request in REQUEST_PROMPTS:
        return REQUEST_PROMPTS[request].format(a=area_id)
    question = request.removeprefix("Custom: ")
    return (f"The user asks about area {area_id}: \"{question}\"\n"
            "Answer this request in `explanation`, based on the slide and, where useful, on your knowledge and "
            "web search. Also transcribe the text of the area; fill the other fields only with what is relevant "
            "to the request (they may be short or empty).")


def request_label(request: str) -> str:
    """Short label of a request, shown in the card title."""
    if request == REQUEST_EXPLAIN:
        return "Explanation"
    if request == REQUEST_RESOURCES:
        return "Additional resources"
    return "“" + textwrap.shorten(request.removeprefix("Custom: "), 60, placeholder="…") + "”"


def step3_detail(state: dict, area_id: str, request: str = REQUEST_EXPLAIN) -> AreaDetail:
    """Step 3 (on demand): information about the area the user clicked on, according to the
    user's request (see REQUEST_CHOICES).

    Two images are sent: a crop of the area (higher effective resolution on its content, so
    small text is read more reliably) and the full slide with the area highlighted (context)."""
    slide = state["image"]
    area = next(a for a in state["structure"]["areas"] if a["id"] == area_id)
    # Crop with a 2% margin so that content slightly outside an imprecise box is not cut off.
    x0, y0, x1, y1 = box_to_px(area["box"], slide.size, margin=0.02)
    crop = slide.crop((int(x0), int(y0), int(x1), int(y1)))
    highlighted = draw_annotations(slide, areas=state["structure"]["areas"],
                                   topics=state["topics"]["topics"], selected=area_id)
    # The full analysis of step 2 is included so the explanation is consistent with it
    # (same topic names, same relations).
    prompt = f"""Step 3 - DETAILS OF AREA {area_id}.
Slide title: {state['structure']['title']!r}
Overall analysis of the slide:
{json.dumps(state['topics'], indent=2, ensure_ascii=False)}

The first image is a crop of area {area_id}; the second is the full slide with that area highlighted.
{request_instructions(request, area_id)}"""
    if STEP_CONFIG["detail"]["web_search"]:
        prompt += """
Use web search to check and enrich the background information with reliable, up-to-date sources
(prefer official documentation, textbooks, encyclopedias and papers), and list the pages you used in `sources`."""
    else:
        prompt += "\nWeb search is not available: return an empty `sources` list."

    response = call_structured(prompt, [crop, highlighted], AreaDetail, step="detail")
    detail = response.output_parsed
    # Add the pages cited by the web search tool that the model did not list itself:
    # the annotations are the reliable record of what was actually consulted.
    listed = {s.url for s in detail.sources}
    detail.sources += [Source(**c) for c in url_citations(response) if c["url"] not in listed]
    return detail

## 4. Drawing and hit-testing

In [ ]:
# One colour per area (cycled if there are more areas than colours).
# The same index -> colour mapping is used for boxes and labels, so they always match.
PALETTE = ["#E63946", "#2A9D8F", "#F4A261", "#457B9D", "#8E44AD",
           "#06A77D", "#D62AD0", "#1D3557", "#B5838D", "#E9C46A"]


def hex_rgb(h: str) -> tuple:
    """'#RRGGBB' -> (R, G, B)."""
    return tuple(int(h[i:i + 2], 16) for i in (1, 3, 5))


def area_color(index: int) -> tuple:
    return hex_rgb(PALETTE[index % len(PALETTE)])


def box_to_px(box: dict, size: tuple, margin: float = 0.0) -> tuple:
    """Normalized 0-1000 box -> pixel box (x0, y0, x1, y1), clamped to the image, optional margin (fraction).

    The model's output is not fully trusted: min/max may be swapped and values may fall
    outside 0-1000, so coordinates are sorted and clamped before use."""
    w, h = size
    x0, x1 = sorted((box["x_min"], box["x_max"]))
    y0, y1 = sorted((box["y_min"], box["y_max"]))
    m = margin * 1000                      # margin expressed in normalized units
    clamp = lambda v: max(0.0, min(1000.0, v))
    return (clamp(x0 - m) * w / 1000, clamp(y0 - m) * h / 1000,
            clamp(x1 + m) * w / 1000, clamp(y1 + m) * h / 1000)


def get_font(size: int):
    """Return a scalable TrueType font if one is available on this system, else Pillow's default."""
    for name in ("DejaVuSans-Bold.ttf", "Arial Bold.ttf", "Arial.ttf", "Helvetica.ttc"):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            pass
    try:
        return ImageFont.load_default(size=size)  # Pillow >= 10.1 has a scalable default font
    except TypeError:
        return ImageFont.load_default()          # older Pillow: small fixed-size bitmap font


def draw_label(draw, xy, text, font, bg, pad):
    """Draw `text` on a filled rectangle whose bottom-left corner is at `xy` (or top-left if no room above)."""
    x, y = xy
    l, t, r, b = draw.textbbox((0, 0), text, font=font)   # exact size of the rendered text
    w, h = r - l + 2 * pad, b - t + 2 * pad
    y = y - h if y - h >= 0 else y  # above the box if there is room, otherwise inside
    draw.rectangle([x, y, x + w, y + h], fill=bg + (235,))
    # Choose black or white text depending on the perceived brightness of the background
    # (ITU-R BT.601 luma), so labels stay readable on light colours such as yellow.
    lum = 0.299 * bg[0] + 0.587 * bg[1] + 0.114 * bg[2]
    draw.text((x + pad - l, y + pad - t), text, font=font, fill=(0, 0, 0) if lum > 160 else (255, 255, 255))


def draw_arrow(draw, p0, p1, color, width):
    """Line from p0 to p1 with a triangular arrowhead at p1."""
    (x0, y0), (x1, y1) = p0, p1
    angle = math.atan2(y1 - y0, x1 - x0)   # direction of the arrow
    head = width * 7                       # arrowhead length scales with the line width
    draw.line([p0, p1], fill=color, width=width)
    # Two points of the head, at +/- 0.4 rad (~23°) from the line direction, behind the tip.
    left = (x1 - head * math.cos(angle - 0.4), y1 - head * math.sin(angle - 0.4))
    right = (x1 - head * math.cos(angle + 0.4), y1 - head * math.sin(angle + 0.4))
    draw.polygon([p1, left, right], fill=color)


def draw_annotations(img, structure=None, areas=None, topics=None, relations=None, selected=None):
    """Return a copy of the slide with title box, area boxes, labels and (optionally) relation arrows.

    Every argument except `img` is optional, so the same function draws the progressive states
    of the UI (only areas -> areas + topics -> + relations -> + selected area).
      structure: dict of step 1 (only its `title_box` is used here)
      areas:     list of area dicts (id, label, box, ...)
      topics:    list of topic dicts of step 2 -> labels show the topic instead of the area label
      relations: list of relation dicts of step 2 -> drawn as arrows between box centres
      selected:  id of the area to highlight
    """
    # Draw on a transparent overlay and composite it on top of the slide: this is how
    # semi-transparent fills are obtained with Pillow (ImageDraw on RGB cannot blend).
    base = img.convert("RGBA")
    overlay = Image.new("RGBA", base.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    W, H = base.size
    # Line width and font size are proportional to the image size, so the result looks the
    # same on a 800px screenshot and on a 4K export.
    lw = max(2, round(min(W, H) / 300))
    font = get_font(max(12, round(min(W, H) / 45)))
    topic_by_id = {t["area_id"]: t for t in (topics or [])}
    areas = areas or []

    # 1) Title: dark outline, no fill (it is not a clickable area).
    if structure and structure.get("title_box"):
        x0, y0, x1, y1 = box_to_px(structure["title_box"], base.size)
        draw.rectangle([x0, y0, x1, y1], outline=(20, 20, 20, 255), width=lw)
        draw_label(draw, (x0, y0), "TITLE", font, (20, 20, 20), lw * 2)

    # 2) Area boxes: light translucent fill; the selected one gets a stronger fill and thicker border.
    for i, a in enumerate(areas):
        color = area_color(i)
        x0, y0, x1, y1 = box_to_px(a["box"], base.size)
        sel = a["id"] == selected
        draw.rectangle([x0, y0, x1, y1], fill=color + ((80 if sel else 28),),
                       outline=color + (255,), width=lw * (3 if sel else 1))

    # 3) Relations: arrows between the centres of the source and target boxes.
    centers = {a["id"]: ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)
               for a in areas for b in [box_to_px(a["box"], base.size)]}
    for rel in relations or []:
        p0, p1 = centers.get(rel["source_area_id"]), centers.get(rel["target_area_id"])
        if not p0 or not p1 or p0 == p1:   # unknown id (model error) or self-relation -> skip
            continue
        # shorten the arrow so it does not cover the centres of the boxes
        s = 0.18
        a0 = (p0[0] + (p1[0] - p0[0]) * s, p0[1] + (p1[1] - p0[1]) * s)
        a1 = (p1[0] - (p1[0] - p0[0]) * s, p1[1] - (p1[1] - p0[1]) * s)
        draw_arrow(draw, a0, a1, (40, 40, 40, 210), lw)
        # Relation type written at the middle of the arrow, with a smaller font.
        mid = ((a0[0] + a1[0]) / 2, (a0[1] + a1[1]) / 2)
        draw_label(draw, mid, rel["relation_type"].replace("_", " "), get_font(max(10, round(min(W, H) / 60))),
                   (250, 250, 250), lw)

    # 4) labels last, so they stay on top of the arrows
    for i, a in enumerate(areas):
        x0, y0, _, _ = box_to_px(a["box"], base.size)
        t = topic_by_id.get(a["id"])
        # Before step 2 the label is the area description; afterwards it is the main topic.
        label = f"{a['id']} · {t['main_topic']}" if t else f"{a['id']} · {a['label']}"
        draw_label(draw, (x0, y0), textwrap.shorten(label, 45, placeholder="…"), font, area_color(i), lw * 2)

    return Image.alpha_composite(base, overlay).convert("RGB")


def find_area_at(x: float, y: float, areas: list, size: tuple) -> Optional[str]:
    """Id of the smallest area containing the pixel (x, y), or None.

    Gradio reports clicks in pixel coordinates of the displayed image, which has the same size
    as the original slide (boxes are drawn at full resolution). When boxes overlap or are nested,
    the smallest one is the most specific, hence the one the user most likely meant."""
    hits = []
    for a in areas:
        x0, y0, x1, y1 = box_to_px(a["box"], size)
        if x0 <= x <= x1 and y0 <= y <= y1:
            hits.append(((x1 - x0) * (y1 - y0), a["id"]))   # (area in px², id)
    return min(hits)[1] if hits else None


FRAME_GREEN = (34, 197, 94)


def framed_crop(img: Image.Image, box: dict, frame_scale: float = 1.10, margin: float = 0.03) -> Image.Image:
    """Crop of one area of the slide, enclosed in a green frame (shown above the area's explanation).

    frame_scale: size of the frame relative to the area's box (1.10 = 10% wider and taller, i.e.
                 5% of breathing space on each side), so the frame does not touch the content.
    margin:      extra slide around the frame (fraction of the slide size), to show some context."""
    W, H = img.size
    x0, y0, x1, y1 = box_to_px(box, img.size)                                         # the box itself
    # Frame: the box enlarged around its centre, clamped to the slide.
    dx, dy = (x1 - x0) * (frame_scale - 1) / 2, (y1 - y0) * (frame_scale - 1) / 2
    fx0, fy0, fx1, fy1 = max(0, x0 - dx), max(0, y0 - dy), min(W, x1 + dx), min(H, y1 + dy)
    # Crop region: the frame plus the margin, clamped to the slide.
    mx, my = margin * W, margin * H
    cx0, cy0 = int(max(0, fx0 - mx)), int(max(0, fy0 - my))
    cx1, cy1 = int(min(W, fx1 + mx)), int(min(H, fy1 + my))
    crop = img.crop((cx0, cy0, cx1, cy1)).convert("RGB")
    bx0, by0, bx1, by1 = fx0, fy0, fx1, fy1                                           # where to draw
    # Line width relative to the crop (not the slide): the crop is shown scaled down to fit the
    # panel, so the frame must be thick enough to stay visible at that size.
    lw = max(4, round(max(crop.size) / 80))
    ImageDraw.Draw(crop).rectangle([bx0 - cx0, by0 - cy0, bx1 - cx0, by1 - cy0], outline=FRAME_GREEN, width=lw)
    return crop

## 5. Loading slide decks (PDF) and single images

A PDF is rendered twice: once as small thumbnails for the slide browser, and — only for the slide
the user selects — at full resolution for display and analysis.

In [ ]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp"}
THUMB_SIDE = 320            # longest side (px) of the thumbnails in the slide browser
RENDER_SIDE = MAX_API_SIDE  # longest side (px) of the selected slide: the resolution sent to the API

# PDFium (the PDF engine of Chrome, used by pypdfium2) is not thread-safe, and Gradio runs
# callbacks in worker threads: all PDF access goes through this lock.
_PDFIUM_LOCK = threading.Lock()


def render_pdf(path: str, max_side: int, pages: Optional[list[int]] = None) -> list[Image.Image]:
    """Render the given pages (0-based; all pages if None) of a PDF so that their longest side is `max_side` px.

    PDF pages have no pixel size, only a size in points (1/72 inch): the render `scale` is the
    factor points -> pixels, chosen per page so that slides of any page format get the same size."""
    with _PDFIUM_LOCK:
        pdf = pdfium.PdfDocument(path)          # opened once for all requested pages
        try:
            out = []
            for i in (range(len(pdf)) if pages is None else pages):
                page = pdf[i]
                w, h = page.get_size()          # in points
                bitmap = page.render(scale=max_side / max(w, h))
                # convert() makes an independent copy, so the image stays valid after the PDF is closed
                out.append(bitmap.to_pil().convert("RGB"))
                page.close()
            return out
        finally:
            pdf.close()


def load_deck(path: str) -> tuple[dict, list[Image.Image]]:
    """Open an uploaded file (PDF deck or single image) -> (deck info, list of thumbnails).

    deck = {"path": file path, "kind": "pdf" | "image", "n_pages": number of slides}
    Only the path is stored (not the rendered pages): slides are rendered on demand in `load_slide`,
    so even a long deck uses little memory."""
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".pdf":
            kind, thumbs = "pdf", render_pdf(path, THUMB_SIDE)
        elif ext in IMAGE_EXTS:
            img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
            img.thumbnail((THUMB_SIDE, THUMB_SIDE))
            kind, thumbs = "image", [img]
        else:
            raise gr.Error(f"Unsupported file type {ext!r}: upload a PDF or a PNG/JPG/WEBP image.")
    except gr.Error:
        raise
    except Exception as e:          # e.g. corrupted or password-protected PDF
        raise gr.Error(f"Could not open the file: {e}")
    if not thumbs:
        raise gr.Error("The PDF has no pages.")
    return {"path": path, "kind": kind, "n_pages": len(thumbs)}, thumbs


def load_slide(deck: dict, index: int) -> Image.Image:
    """Full-resolution image of slide `index` (0-based), used for display, clicks and analysis."""
    if deck["kind"] == "pdf":
        return render_pdf(deck["path"], RENDER_SIDE, [index])[0]
    # Single image: keep its original resolution (it is downscaled only when sent to the API).
    # EXIF orientation is applied so that phone photos are not shown rotated.
    return ImageOps.exif_transpose(Image.open(deck["path"])).convert("RGB")

## 6. Markdown rendering of the results

In [ ]:
def overview_md(state: dict) -> str:
    """Overview panel: title, overall message, table of areas/topics and table of relations.

    Works with partial results (e.g. after step 1 only), so it can be refreshed after each step."""
    s, topics = state.get("structure"), state.get("topics")
    if not s:
        return ""
    out = [f"## {s['title']}" + (" *(inferred)*" if s["title_is_inferred"] else "")]
    if s.get("subtitle"):
        out.append(f"*{s['subtitle']}*")
    if topics:
        out.append(f"\n**Overall message:** {topics['overall_message']}")
    # Areas table: "…" placeholders are shown until step 2 has filled in topic and role.
    by_id = {x["area_id"]: x for x in (topics or {}).get("topics", [])}
    out.append("\n### Areas\n| Area | Topic | Role | Content |\n|---|---|---|---|")
    for a in s["areas"]:
        tp = by_id.get(a["id"], {})
        out.append(f"| **{a['id']}** | {tp.get('main_topic', '…')} | {tp.get('role', '…').replace('_', ' ')} "
                   f"| {a['content_type'].replace('_', ' ')} |")
    if topics and topics["relations"]:
        out.append("\n### Relations\n| From | Relation | To | Why |\n|---|---|---|---|")
        for r in topics["relations"]:
            out.append(f"| {r['source_area_id']} | *{r['relation_type'].replace('_', ' ')}* | "
                       f"{r['target_area_id']} | {r['description']} |")
    return "\n".join(out)


def detail_md(d: dict) -> str:
    """Body of the explanation of one area (output of step 3), as Markdown. Empty sections are omitted.
    The heading (area id + topic) and the transcribed text are added by `area_card`."""
    out = [d["explanation"]]
    if d["key_points"]:
        out.append("### Key points\n" + "\n".join(f"- {p}" for p in d["key_points"]))
    if d["background"]:
        out.append(f"### Background (not on the slide)\n{d['background']}")
    if d["connections"]:
        out.append("### Connections\n" + "\n".join(f"- **{c['area_id']}**: {c['how_related']}"
                                                   for c in d["connections"]))
    if d["questions_to_explore"]:
        out.append("### Questions to explore\n" + "\n".join(f"- {q}" for q in d["questions_to_explore"]))
    if d["sources"]:
        out.append("### Sources\n" + "\n".join(f"- [{s['title']}]({s['url']})" for s in d["sources"]))
    return "\n\n".join(out)


# --- details panel: one collapsible card per clicked area ------------------------------------
# The panel is a gr.HTML component holding a list of <details> elements (the browser's native
# collapsible block). The cards accumulate over the whole deck, in the order the areas were clicked:
# a click appends a new card, opened, and collapses the previous ones; moving to another slide or
# analyzing one collapses all cards but keeps them. Collapsed cards re-open by clicking their title.
# The cards of the other slides are grouped in a collapsible "Previous slides" section at the top;
# the cards of the selected slide follow. (The downloadable report has no such section.)

# Markdown -> HTML converter. html=False: raw HTML inside the model's text is shown as text, not
# executed (the text comes from the model and from web pages it searched).
# linkify: plain web addresses in the text (not written as [text](url)) become links too. It needs
# the optional package linkify-it-py; without it, only [text](url) links are links.
_LINKIFY = importlib.util.find_spec("linkify_it") is not None
_md = MarkdownIt("commonmark", {"html": False, "linkify": _LINKIFY}).enable("table")   # tables: overview in the report
if _LINKIFY:
    _md.enable("linkify")


def _link_in_new_tab(self, tokens, idx, options, env):
    """Render every link with target="_blank", so it opens in a new browser tab (and does not
    replace the app); rel="noopener noreferrer" stops the opened page from accessing the app's tab."""
    tokens[idx].attrSet("target", "_blank")
    tokens[idx].attrSet("rel", "noopener noreferrer")
    return self.renderToken(tokens, idx, options, env)


_md.add_render_rule("link_open", _link_in_new_tab)

NO_DETAIL_TEXT = "Click on a box to see more information about its topic."
MAX_CROP_SIDE = 900   # px; crops are embedded in the page as base64 images, so keep them small


def crop_data_url(analysis: dict, area_id: str) -> str:
    """Framed crop of an area as a base64 'data:' URL (cached in the analysis: computed once per area)."""
    cache = analysis.setdefault("crop_urls", {})
    if area_id not in cache:
        area = next(a for a in analysis["structure"]["areas"] if a["id"] == area_id)
        crop = framed_crop(analysis["image"], area["box"])
        crop.thumbnail((MAX_CROP_SIDE, MAX_CROP_SIDE))
        cache[area_id] = to_data_url(crop)
    return cache[area_id]


def area_card(analysis: dict, area_id: str, request: str, is_open: bool, loading: bool) -> str:
    """HTML of one card: title (slide number, area id, topic, request), framed crop, the text found in
    the area (collapsed), then the answer (or a spinner while the answer is being generated)."""
    d = analysis["details"].get((area_id, request))   # answers are cached per (area, request)
    topic = d["topic"] if d else next((t["main_topic"] for t in analysis["topics"]["topics"]
                                       if t["area_id"] == area_id), "")
    text = ""
    if loading or not d:
        body = f"<p>{THROBBER}Loading details for <b>{area_id}</b>…</p>"
    else:
        body = _md.render(detail_md(d))
        # The transcribed text, right below the image it comes from; it is long and rarely needed,
        # so it is a nested collapsible block.
        if d["extracted_text"]:
            text = ("<details class='area-text'><summary>Text found in the area</summary>"
                    f"<pre>{html.escape(d['extracted_text'])}</pre></details>")
    # The title starts with the slide number (position in the deck), so that every explanation can be
    # traced back to its slide, also in the downloaded report: "Slide 31: A2 · Markov assumption".
    slide = f"Slide {analysis['page'] + 1}: " if "page" in analysis else ""
    return (f"<details class='area-card'{' open' if is_open else ''}>"
            f"<summary>{slide}<b>{area_id}</b> · {html.escape(topic)}"
            f" <span class='card-request'>— {html.escape(request_label(request))}</span></summary>"
            f"<div class='area-card-body'>"
            f"<img class='area-crop' src='{crop_data_url(analysis, area_id)}' alt='Area {area_id}'>"
            f"{text}{body}</div></details>")


def details_html(session: Optional[dict], open_last: bool = False, loading: Optional[tuple] = None,
                 hint: Optional[str] = None) -> str:
    """HTML of the whole details panel: one card per clicked area and request, in click order
    (session["history"], a list of (slide index, area id, request)). Cards of the other slides are grouped
    in a collapsed "Previous slides" section; then an optional hint; then the cards of the selected slide.

    open_last: open the last card (right after a click); otherwise all cards are collapsed.
    loading:   (slide index, area id, request) of a card whose answer is still being generated."""
    history = (session or {}).get("history", [])
    current = (session or {}).get("current")
    previous, here = [], []
    for i, (page, area_id, request) in enumerate(history):
        analysis = session["analyses"].get(page)
        if analysis is None:
            continue
        card = area_card(analysis, area_id, request, is_open=open_last and i == len(history) - 1,
                         loading=(page, area_id, request) == loading)
        (here if page == current else previous).append(card)
    parts = []
    if previous:
        parts.append(f"<details class='previous-slides'><summary>Previous slides ({len(previous)} "
                     f"card{'s' if len(previous) > 1 else ''})</summary>"
                     f"<div class='previous-slides-body'>{''.join(previous)}</div></details>")
    if hint or not here:
        parts.append(f"<p class='area-hint'>{hint or NO_DETAIL_TEXT}</p>")
    parts += here
    return "\n".join(parts)


# --- downloadable report -------------------------------------------------------------------
# A single self-contained HTML file (images embedded as base64), with every analyzed slide of the
# deck: annotated slide, segmentation results and the explanations that were opened, plus a pane
# on the left listing the slides (number + title) to jump to any of them.
# To get a PDF, open the file in a browser and use Print -> Save as PDF (each slide starts on a new page).
# The cards are collapsed in the file; a small script expands all of them while printing (a collapsed
# <details> element would otherwise be printed collapsed) and restores them afterwards.

REPORT_SCRIPT = """
let closedBeforePrint = [];
window.addEventListener('beforeprint', () => {
  closedBeforePrint = [...document.querySelectorAll('details:not([open])')];
  closedBeforePrint.forEach(d => d.open = true);
});
window.addEventListener('afterprint', () => closedBeforePrint.forEach(d => d.open = false));

// Left pane: highlight the entry of the topmost slide visible in the window.
window.addEventListener('DOMContentLoaded', () => {
  const sections = [...document.querySelectorAll('section.slide')];
  const links = new Map([...document.querySelectorAll('nav.toc a')].map(a => [a.getAttribute('href').slice(1), a]));
  const visible = new Set();
  const observer = new IntersectionObserver(entries => {
    entries.forEach(e => e.isIntersecting ? visible.add(e.target.id) : visible.delete(e.target.id));
    const first = sections.find(s => visible.has(s.id));
    if (first) links.forEach((a, id) => a.classList.toggle('active', id === first.id));
  });
  sections.forEach(s => observer.observe(s));
});
"""

REPORT_CSS = """
body { font-family: -apple-system, 'Segoe UI', Helvetica, Arial, sans-serif; color: #27272a;
       margin: 0; line-height: 1.45; }
/* left pane: list of the slides, fixed while the content scrolls */
nav.toc { position: fixed; top: 0; bottom: 0; left: 0; width: 250px; box-sizing: border-box; overflow-y: auto;
          padding: 1.2em 0.8em; background: #f4f4f5; border-right: 1px solid #e4e4e7; }
nav.toc h2 { font-size: 0.8em; text-transform: uppercase; letter-spacing: 0.05em; color: #71717a; margin: 0 0 0.6em 0.5em; }
nav.toc a { display: block; padding: 6px 8px; margin-bottom: 2px; border-radius: 6px; color: inherit; text-decoration: none; }
nav.toc a:hover { background: #e4e4e7; }
nav.toc a.active { background: #e4e4e7; font-weight: 600; }
nav.toc .toc-num { display: block; font-size: 0.8em; color: #71717a; font-weight: normal; }
main { margin-left: 250px; max-width: 960px; padding: 2em 2em 4em 2em; }
section.slide { scroll-margin-top: 1em; }
@media (max-width: 800px) {   /* narrow screens: the list goes on top of the page */
  nav.toc { position: static; width: auto; border-right: none; border-bottom: 1px solid #e4e4e7; }
  main { margin-left: 0; padding: 1em; }
}
h1 { margin-bottom: 0.2em; } .meta { color: #71717a; margin-top: 0; }
section.slide { border-top: 2px solid #e4e4e7; margin-top: 2em; padding-top: 1em; }
img.slide-img { display: block; max-width: 100%; border: 1px solid #e4e4e7; border-radius: 6px; margin: 0.5em 0 1em 0; }
table { border-collapse: collapse; margin: 0.5em 0; } th, td { border: 1px solid #a1a1aa; padding: 4px 10px; text-align: left; }
.area-card { background-color: #e4e4e7; border-radius: 8px; margin-bottom: 10px; }
.area-card > summary { cursor: pointer; padding: 10px 14px; font-size: 1.15em; }
.card-request { color: #71717a; font-size: 0.85em; }
.area-card-body { padding: 0 16px 12px 16px; }
.area-card img.area-crop { display: block; max-width: 100%; max-height: 260px; margin: 4px auto 12px auto; }
.area-text { margin-bottom: 0.8em; }
.area-text pre { white-space: pre-wrap; font-size: 0.85em; }
@media print {
  nav.toc { display: none; }
  main { margin-left: 0; padding: 0; }
  section.slide { break-before: page; border-top: none; }
  .area-card { break-inside: avoid; }
}
"""


def report_html(session: dict) -> str:
    """HTML of the downloadable report for all analyzed slides of the session's deck."""
    deck = session["deck"]
    name = os.path.basename(deck["path"])
    # Left pane: one link per analyzed slide (number + title), pointing to the section's id.
    toc = ["<nav class='toc'><h2>Slides</h2>"]
    for page in sorted(session["analyses"]):
        title = session["analyses"][page]["structure"]["title"]
        toc.append(f"<a href='#slide-{page + 1}'><span class='toc-num'>Slide {page + 1}</span>"
                   f"{html.escape(title)}</a>")
    toc.append("</nav>")
    parts = [f"<h1>Slide analysis – {html.escape(name)}</h1>",
             f"<p class='meta'>Generated on {datetime.now():%Y-%m-%d %H:%M} · "
             f"{len(session['analyses'])} of {deck['n_pages']} slide(s) analyzed · models: "
             + ", ".join(f"{step} = {cfg['model']}" for step, cfg in STEP_CONFIG.items()) + "</p>"]
    for page in sorted(session["analyses"]):
        analysis = session["analyses"][page]
        explained = [(a, r) for p, a, r in session.get("history", []) if p == page and (a, r) in analysis["details"]]
        parts.append(f"<section class='slide' id='slide-{page + 1}'><h2>Slide {page + 1} of {deck['n_pages']}</h2>")
        # annotated slide, with the relations drawn as arrows
        parts.append(f"<img class='slide-img' src='{to_data_url(render(analysis, True))}' alt='Slide {page + 1}'>")
        parts.append(_md.render(overview_md(analysis)))
        parts.append("<h3>Explained areas</h3>" if explained else
                     "<p><i>No area was explained for this slide.</i></p>")
        # cards collapsed: the reader opens the ones of interest (they are expanded for printing, see below)
        parts += [area_card(analysis, a, r, is_open=False, loading=False) for a, r in explained]
        parts.append("</section>")
    return ("<!doctype html><html><head><meta charset='utf-8'>"
            f"<title>Slide analysis – {html.escape(name)}</title><style>{REPORT_CSS}</style>"
            f"<script>{REPORT_SCRIPT}</script></head><body>"
            + "".join(toc) + "<main>" + "\n".join(parts) + "</main></body></html>")

## 7. Gradio callbacks

Navigation (`go_to`) shows the selected slide, restoring its cached analysis if it was analyzed before.
`analyze_slide` is a generator: after every step it yields, so the UI updates progressively (title + areas → topics).

In [ ]:
# All data of one browser session is kept in a plain dict stored in a gr.State (one per browser
# tab, so several users/tabs do not interfere):
#   session = {
#       "deck":     {"path", "kind", "n_pages"}       <- the uploaded file (see load_deck)
#       "current":  0-based index of the selected slide
#       "image":    PIL image of the selected slide (full resolution)
#       "analyses": {slide index: analysis}           <- cache: going back to an analyzed slide
#                                                        costs no API call
#       "history":  [(slide index, area id, request)]  <- clicked areas of the whole deck, in click
#   }                                                    order = the cards of the details panel
# and every analysis (one per slide) is:
#   analysis = {
#       "page":      0-based index of the slide in the deck (shown as "Slide N" on the cards),
#       "image":     PIL image of that slide,
#       "structure": step 1 output as dict (title fields + "areas"),
#       "topics":    step 2 output as dict,
#       "details":   {(area_id, request): step 3 output as dict}  <- cache, filled on click
#   }
# Plain dicts (model_dump()) are used instead of Pydantic objects because they can be shown
# directly in gr.JSON and passed to the drawing functions.
#
# Most callbacks return the same group of values ("page outputs", built by `view`):
#   [annotated image, status, overview, area dropdown, details, session]
# and the navigation callbacks add the slide slider in front of them.

# Animated spinner shown in progress messages; it is a plain HTML element styled by the
# `.throbber` CSS rule passed to demo.launch() (see section 8).
THROBBER = '<span class="throbber"></span>'



def current_analysis(session) -> Optional[dict]:
    """Analysis of the selected slide, or None if it has not been analyzed yet."""
    if not session:
        return None
    return session["analyses"].get(session["current"])


def slide_label(session) -> str:
    return f"Slide {session['current'] + 1}/{session['deck']['n_pages']}"


def render(analysis: dict, show_relations: bool, selected: Optional[str] = None) -> Image.Image:
    """Draw the annotated slide from whatever is currently available in `analysis`."""
    return draw_annotations(
        analysis["image"],
        structure=analysis.get("structure"),
        areas=(analysis.get("structure") or {}).get("areas"),
        topics=(analysis.get("topics") or {}).get("topics"),
        relations=(analysis.get("topics") or {}).get("relations") if show_relations else None,
        selected=selected,
    )


def view(session, analysis: Optional[dict], show_relations: bool, status: str) -> tuple:
    """Page outputs for the selected slide: the plain slide if `analysis` is None, else the annotated one.
    The details panel keeps all the cards of the deck, collapsed."""
    image = render(analysis, show_relations) if analysis else session["image"]
    structure = analysis["structure"] if analysis else None
    topics = analysis["topics"] if analysis else None
    choices = [a["id"] for a in structure["areas"]] if structure else []
    return (image, status, overview_md(analysis) if analysis else "",
            gr.update(choices=choices, value=None), details_html(session), session)


# --- navigation in the deck -----------------------------------------------------------

def go_to(session, page, show_relations):
    """Select slide `page` (1-based, as shown on the slider) and show it.

    If the slide was analyzed before, its cached analysis is shown (no API call);
    otherwise the slide is rendered at full resolution and waits for 'Analyze slide'.
    Returns: [slider, *page outputs]."""
    if not session:
        return (gr.update(), *[gr.update()] * 5, session)
    n = session["deck"]["n_pages"]
    i = min(max(int(page), 1), n) - 1          # clamp: prev on the first / next on the last slide
    session["current"] = i
    analysis = session["analyses"].get(i)
    if analysis:
        session["image"] = analysis["image"]
        status = f"✅ {slide_label(session)} – already analyzed, click on a box for details."
    else:
        session["image"] = load_slide(session["deck"], i)
        status = f"{slide_label(session)} – press **Analyze slide** to analyze it."
    # Gradio requires minimum < maximum: a one-slide deck keeps maximum=2 with the slider disabled.
    slider = gr.update(value=i + 1, maximum=max(n, 2), interactive=n > 1)
    return (slider, *view(session, analysis, show_relations, status))


def new_session(path: str) -> tuple[dict, list[Image.Image]]:
    """Fresh session for an uploaded file (no slide selected yet) + the thumbnails of its slides."""
    deck, thumbs = load_deck(path)
    return {"deck": deck, "current": 0, "image": None, "analyses": {}, "history": []}, thumbs


def on_upload(path, show_relations):
    """A file was uploaded (or removed): load the deck, fill the slide browser, show slide 1.
    Returns: [gallery, analyze button, slider, *page outputs].
    The Analyze button is enabled only once a slide is loaded, so it cannot be pressed
    while the upload is still being processed."""
    if path is None:   # file removed -> reset everything
        return (gr.update(value=[]), gr.update(interactive=False),
                gr.update(value=1, maximum=2, interactive=False),
                None, "", "", gr.update(choices=[], value=None), details_html(None), None)
    session, thumbs = new_session(path)
    gallery = [(t, str(i + 1)) for i, t in enumerate(thumbs)]   # (image, caption = slide number)
    return (gallery, gr.update(interactive=True), *go_to(session, 1, show_relations))


def on_gallery_select(session, show_relations, evt: gr.SelectData):
    """Click on a thumbnail: `evt.index` is the 0-based position of the thumbnail in the gallery."""
    return go_to(session, evt.index + 1, show_relations)


# --- analysis of the selected slide -----------------------------------------------------

def analyze_slide(session, path, page, show_relations):
    """Callback of the 'Analyze slide' button: runs steps 1-2 on the selected slide.

    It is a *generator*: every `yield` immediately pushes an update to the browser, so the
    user sees the boxes after step 1 while step 2 is still running.
    Pressing the button on an already analyzed slide runs the analysis again (e.g. after
    changing STEP_CONFIG or if the boxes were poor); just navigating back to it does not."""
    if not session or session.get("image") is None:
        # The session can be empty even though a file is shown in the upload box, e.g. when the
        # page was reloaded (a reload starts a new Gradio session with empty gr.State values).
        # In that case rebuild it from the uploaded file and the slide shown on the slider.
        if not path:
            raise gr.Error("Please upload a slide deck (PDF) or a slide image first.")
        session, _ = new_session(path)
        session = go_to(session, page, show_relations)[-1]
    page = session["current"]
    label = slide_label(session)
    # The analysis is built in a local dict and stored in the cache only when complete,
    # so a failed analysis never looks like a finished one.
    analysis = {"page": page, "image": session["image"], "structure": None, "topics": None, "details": {}}
    try:
        yield view(session, analysis, show_relations, f"{THROBBER} {label} · step 1/2 – identifying the title and the relevant areas…")
        structure = step1_structure(analysis["image"])
        analysis["structure"] = structure.model_dump()
        yield view(session, analysis, show_relations, f"{THROBBER} {label} · step 2/2 – identifying topics and relations…")
        analysis["topics"] = step2_topics(analysis["image"], structure).model_dump()
    except Exception as e:
        # gr.Error is shown as a toast in the UI instead of a stack trace.
        raise gr.Error(f"Analysis failed: {e}")
    session["analyses"][page] = analysis
    # Cards of a previous analysis of this slide described areas that may no longer exist: drop them.
    session["history"] = [e for e in session.get("history", []) if e[0] != page]
    yield view(session, analysis, show_relations,
               f"✅ {label} – {len(analysis['structure']['areas'])} areas found. Click on a box for details.")


# --- details of an area ------------------------------------------------------------------

def show_detail(session, area_id, show_relations, choice=REQUEST_EXPLAIN, custom_text=""):
    """Generator: highlight the area immediately, then fetch (or reuse cached) the answer to the
    request chosen in "What would you like to know?" (choice + custom_text, see request_key).

    The (area, request) pair is appended to the deck's click history, so the details panel gets a new
    card at the bottom (opened) while all the previous cards, of any slide, are collapsed. Asking the
    same thing about the same area again moves that card to the bottom instead of duplicating it.
    Used both by clicks on the image and by the area dropdown.
    Yields values for: [annotated image, details panel, dropdown, session]."""
    analysis = current_analysis(session)
    if not analysis or not area_id:
        # gr.update() with no arguments = "leave this component unchanged".
        yield gr.update(), details_html(session, hint="Analyze this slide first."), gr.update(), session
        return
    request = request_key(choice, custom_text)
    if request is None:   # "Custom" chosen but the text box is empty
        yield gr.update(), details_html(session, hint="Write your request in the box first."), gr.update(), session
        return
    history = session.setdefault("history", [])
    entry = (session["current"], area_id, request)
    if entry in history:
        history.remove(entry)
    history.append(entry)
    image = render(analysis, show_relations, selected=area_id)
    if (area_id, request) not in analysis["details"]:
        # First click on this area: show the highlight + the new card with a spinner right away,
        # then call the model (this can take several seconds).
        yield image, details_html(session, open_last=True, loading=entry), area_id, session
        try:
            analysis["details"][(area_id, request)] = step3_detail(analysis, area_id, request).model_dump()
        except Exception as e:
            history.remove(entry)   # no card for an area whose details could not be loaded
            raise gr.Error(f"Could not load details: {e}")
    # Cached (or just fetched) result: no further API call for repeated clicks.
    yield image, details_html(session, open_last=True), area_id, session


def on_image_click(session, show_relations, choice, custom_text, evt: gr.SelectData):
    """Callback of a click on the annotated image.

    Gradio injects `evt` automatically because of the gr.SelectData type hint;
    for images, `evt.index` is the [x, y] pixel position of the click."""
    analysis = current_analysis(session)
    if not analysis:
        yield gr.update(), details_html(session, hint="Analyze this slide first."), gr.update(), session
        return
    x, y = evt.index
    area_id = find_area_at(x, y, analysis["structure"]["areas"], analysis["image"].size)
    if area_id is None:
        # Click outside every box: remove any highlight and tell the user.
        yield (render(analysis, show_relations),
               details_html(session, hint="No area here – click inside one of the coloured boxes."),
               None, session)
        return
    yield from show_detail(session, area_id, show_relations, choice, custom_text)


def update_report(session):
    """Refresh the file behind the 'Download report' button (run after every analysis or explanation).

    The file is written in a temporary folder of this session (Gradio serves files from the system
    temp folder) and named after the uploaded deck. The button is disabled while there is nothing to report."""
    if not session or not session["analyses"]:
        return gr.update(value=None, interactive=False)
    folder = session.setdefault("report_dir", tempfile.mkdtemp(prefix="slide_report_"))
    os.makedirs(folder, exist_ok=True)   # recreate it if the system cleaned up the temp folder
    stem = os.path.splitext(os.path.basename(session["deck"]["path"]))[0]
    path = os.path.join(folder, f"{stem}_analysis.html")
    with open(path, "w", encoding="utf-8") as f:
        f.write(report_html(session))
    return gr.update(value=path, interactive=True)


def on_toggle_relations(session, show_relations, area_id):
    """Redraw the image when the 'Show relations' checkbox changes (no API call)."""
    analysis = current_analysis(session)
    if not analysis:
        return gr.update()      # slide not analyzed: nothing drawn on it, nothing to change
    return render(analysis, show_relations, selected=area_id)

## 8. UI

In [ ]:
# Custom CSS (passed to demo.launch): a small spinning circle used as progress indicator.
# The colours come from Gradio's theme variables, so it matches light and dark mode.
# The border is written side by side (not with the `border`/`border-color` shorthands): Gradio
# rewrites custom CSS to scope it to the app, and that rewrite drops shorthands containing var(...).
# `!important`: Gradio's base stylesheet resets the border width of all elements to 0.
CSS = """
.throbber {
    display: inline-block; width: 1em; height: 1em; margin-right: 0.4em; vertical-align: -0.15em;
    border-width: 2px !important; border-style: solid !important;
    border-top-color: var(--color-accent) !important;
    border-right-color: var(--border-color-primary) !important;
    border-bottom-color: var(--border-color-primary) !important;
    border-left-color: var(--border-color-primary) !important;
    border-radius: 50%; animation: throbber-spin 0.8s linear infinite;
}
@keyframes throbber-spin { to { transform: rotate(360deg); } }
/* Steps 1-3: boxes with a thin border and the numbered title on top (border written side by side:
   see the note on shorthands above). */
.step-box {
    border-radius: 10px; padding: 10px 14px !important;
    border-width: 1px !important; border-style: solid !important;
    border-top-color: var(--border-color-primary) !important; border-right-color: var(--border-color-primary) !important;
    border-bottom-color: var(--border-color-primary) !important; border-left-color: var(--border-color-primary) !important;
}
.step-box h3 { margin-top: 0 !important; }
/* Centered text in the step-3 and step-5 boxes (titles and status line). */
.step-center { text-align: center; }
/* Request shown in the card title, e.g. "— Additional resources". */
.card-request { color: var(--body-text-color-subdued); font-size: 0.85em; }
/* Analyze and Download buttons: fixed width, centered in their box at any window width. */
.center-btn { width: 280px !important; max-width: 100% !important; align-self: center !important;
              flex-grow: 0 !important; margin-left: auto !important; margin-right: auto !important; }
/* Details panel: one grey card per clicked area (a <details> element, collapsible). */
.area-card {
    /* same grey as the panels of the segmentation results (Gradio paints groups with this colour) */
    background-color: var(--border-color-primary); border-radius: 8px; margin-bottom: 10px;
}
.area-card > summary { cursor: pointer; padding: 10px 14px; font-size: 1.15em; }
.area-card-body { padding: 0 16px 12px 16px; }
.area-card img.area-crop { display: block; max-width: 100%; max-height: 260px; margin: 4px auto 12px auto; }
.area-card .area-text { margin-bottom: 0.8em; }
.area-card .area-text pre { white-space: pre-wrap; }
.area-hint { font-style: italic; color: var(--body-text-color-subdued); }
/* Collapsible group with the cards of the other slides (indented, with a line on the left). */
.previous-slides { margin-bottom: 12px; }
.previous-slides > summary { cursor: pointer; padding: 6px 2px; font-weight: 600; color: var(--body-text-color-subdued); }
.previous-slides-body { padding-left: 12px; margin-top: 6px;
    border-left-width: 3px !important; border-left-style: solid !important;
    border-left-color: var(--border-color-primary) !important; }
"""

with gr.Blocks(title="Slide Analyzer") as demo:
    gr.Markdown("# 🖼️ Slide Analyzer\nUpload a slide deck (PDF) or a slide image, pick a slide, press "
                "**Analyze slide**, then click on a box to learn more about it.")
    # Per-session storage of the deck and of all analyses (see the comment at the top of section 7).
    session = gr.State(None)

    # --- row 1: step 1 (file upload) | step 2 (slide navigation) ------------------------------
    # Each step is a box with a thin border (CSS class "step-box") and a numbered title on top,
    # to guide the user through the app.
    with gr.Row(equal_height=True):
        with gr.Column(scale=2, elem_classes=["step-box"]):
            gr.Markdown("### 1. Upload the slide deck")
            # type="filepath" -> callbacks receive the path of the uploaded (temporary) file.
            file_in = gr.File(label="Slide deck (PDF) or slide image", type="filepath",
                              file_types=[".pdf", *sorted(IMAGE_EXTS)], height=140)
        with gr.Column(scale=3, elem_classes=["step-box"]):
            gr.Markdown("### 2. Choose a slide")
            # Navigation: the slider is the "current slide" indicator; prev/next move by one.
            # (Clicking a thumbnail in the strip below also selects a slide.)
            # Gradio requires minimum < maximum, hence maximum=2 until a deck is loaded (see go_to).
            with gr.Row():
                prev_btn = gr.Button("◀ Prev", scale=0, min_width=90)
                slider = gr.Slider(minimum=1, maximum=2, step=1, value=1, label="Slide", interactive=False)
                next_btn = gr.Button("Next ▶", scale=0, min_width=90)
            show_rel = gr.Checkbox(label="Show relations as arrows", value=True)

    # --- step 3: Analyze button (left) | what to ask about a clicked area (right) ---------------
    with gr.Column(elem_classes=["step-box"]):
        gr.Markdown("### 3. Analyze the slide: identify title and topics")
        with gr.Row(equal_height=False):
            # Left: 1/4 of the width (button + status); right: 3/4 (request menu).
            with gr.Column(scale=1, min_width=200):
                # Disabled until a slide is loaded (enabled by on_upload).
                analyze_btn = gr.Button("Analyze slide", variant="primary", interactive=False)
                status = gr.Markdown()     # progress / navigation messages
            with gr.Column(scale=3, min_width=260):
                # The request is sent to the model, with the area, when the user clicks on a box.
                gr.Markdown("**What would you like to know?**")
                request_dd = gr.Dropdown(REQUEST_CHOICES, value=REQUEST_EXPLAIN, show_label=False, interactive=True)
                # Shown only when "Custom" is selected (see request_dd.change below).
                custom_tb = gr.Textbox(show_label=False, lines=2, visible=False,
                                       placeholder="Write your request, then click on a box in the slide")

    # --- row 2: vertical thumbnail strip (like the sidebar of macOS Preview) | selected slide --
    SLIDE_HEIGHT = 560   # px; the thumbnail strip has the same height and scrolls
    with gr.Row(equal_height=False):
        with gr.Column(scale=1, min_width=170):
            # columns=1 -> one thumbnail per row = a vertical strip. allow_preview=False: a click
            # selects the slide instead of opening Gradio's full-screen preview.
            gallery = gr.Gallery(label="Slides", columns=1, height=SLIDE_HEIGHT,
                                 allow_preview=False, object_fit="contain")
        with gr.Column(scale=6):
            # interactive=False: output-only image (no upload), but it still emits `select` on click.
            # show_label=False: the label badge would cover the top of the slide, where the title is.
            annotated = gr.Image(label="Selected slide", show_label=False, type="pil",
                                 interactive=False, height=SLIDE_HEIGHT)

    # --- row 3: results of the segmentation (title, areas, relations) ----------------------
    # Each of rows 3 and 4 is a bordered panel (gr.Group), so the two texts are visually separate.
    with gr.Group():
        # Alternative to clicking (useful for small or overlapping boxes, and for accessibility).
        area_dd = gr.Dropdown(label="Pick an area (or click on a box in the slide)", choices=[], interactive=True)
        overview = gr.Markdown(padding=True)

    # --- step 4: explanations of the clicked areas (title only, no box) --------------------------
    # One collapsible card per clicked area (see details_html): the area cut out of the slide in a
    # green frame, then its explanation. New cards are appended at the bottom.
    gr.Markdown("### 4. Check out the information cards")
    details = gr.HTML(details_html(None))

    # --- step 5: download, in a box, button centered ---------------------------------------------
    # Downloads the HTML report of all analyzed slides; its file is refreshed by update_report
    # after every analysis / explanation (see the `.then` calls below).
    with gr.Column(elem_classes=["step-box"]):
        gr.Markdown("### 5. Download the cards as report", elem_classes=["step-center"])
        report_btn = gr.DownloadButton("Download", interactive=False, elem_classes=["center-btn"])

    # --- event wiring --------------------------------------------------------------------
    # The order of the outputs must match the order of the values returned/yielded by the callbacks.
    page_outputs = [annotated, status, overview, area_dd, details, session]
    nav_outputs = [slider, *page_outputs]

    # Upload / removal of the file -> rebuild the slide browser and show slide 1.
    # `.then(...)` runs a second function after the first one has finished: here it refreshes the
    # report file (or disables the download button when there is nothing to report).
    file_in.change(on_upload, [file_in, show_rel], [gallery, analyze_btn, *nav_outputs]).then(
        update_report, session, report_btn)

    # Navigation. Each event calls go_to() explicitly instead of relying on slider.change, so that
    # a programmatic slider update (e.g. from prev/next) does not trigger a second render.
    # `.release` fires when the user lets go of the slider handle (not on every intermediate value).
    gallery.select(on_gallery_select, [session, show_rel], nav_outputs)
    slider.release(go_to, [session, slider, show_rel], nav_outputs)
    prev_btn.click(lambda s, p, r: go_to(s, p - 1, r), [session, slider, show_rel], nav_outputs)
    next_btn.click(lambda s, p, r: go_to(s, p + 1, r), [session, slider, show_rel], nav_outputs)

    # Analysis of the selected slide (steps 1-2). The file and the slider are passed too, so that
    # the session can be rebuilt if it was lost (see analyze_slide).
    analyze_btn.click(analyze_slide, [session, file_in, slider, show_rel], page_outputs).then(
        update_report, session, report_btn)

    # Details (step 3): click on the annotated image -> hit-test -> details; or pick from the dropdown.
    detail_outputs = [annotated, details, area_dd, session]
    annotated.select(on_image_click, [session, show_rel, request_dd, custom_tb], detail_outputs).then(
        update_report, session, report_btn)
    # `.input` (not `.change`) fires only on user interaction: when a click on the image sets the
    # dropdown value programmatically, this does NOT trigger a second, duplicate detail request.
    area_dd.input(show_detail, [session, area_dd, show_rel, request_dd, custom_tb], detail_outputs).then(
        update_report, session, report_btn)
    # "Custom" in the request menu shows the text box for the user's own request.
    request_dd.change(lambda choice: gr.update(visible=(choice == REQUEST_CUSTOM)), request_dd, custom_tb)
    show_rel.change(on_toggle_relations, [session, show_rel, area_dd], annotated)

In [ ]:
# Start the web server and show the app inside the notebook output.
# Use demo.launch(share=True) for a temporary public link, or open the printed local URL in a browser.
# Re-running this cell after code changes: call demo.close() first (or restart the kernel).
demo.launch(inline=True, height=1100, css=CSS, share=True)